# Example notebook

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from mudata import AnnData, MuData
import perturbvi
import pandas as pd
import pyro
import torch 

perturbvi.__version__

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Global seed set to 0


'0.0.1'

In [16]:
rna_key = "rna"
perturb_key = "grna"
n_cells = 1000

total_rna = pd.DataFrame({'lib_size':np.random.lognormal(10, 1, size=(n_cells))})
rna_counts = np.random.negative_binomial(100, 0.9, size=(n_cells, 10))
rna_adata = AnnData(rna_counts, obs=total_rna, dtype=np.float64)

rna_adata.X

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


array([[17., 13., 16., ..., 10., 14.,  9.],
       [ 9.,  9., 13., ...,  9., 15.,  8.],
       [ 7., 13., 13., ..., 14., 13., 12.],
       ...,
       [18.,  4.,  9., ..., 12.,  9., 11.],
       [11.,  7.,  5., ..., 14.,  8., 16.],
       [ 8., 14., 13., ..., 12., 14.,  9.]])

In [4]:
perturb_adata = AnnData(np.random.binomial(1, 0.5, size=(n_cells, 5)), dtype=np.float64)
perturb_adata.var_names = 'guide' + perturb_adata.var_names
mdata = MuData({rna_key: rna_adata, perturb_key: perturb_adata})
mdata

MuData object with n_obs × n_vars = 1000 × 15
  2 modalities
    rna:	1000 x 10
      obs:	'lib_size'
    grna:	1000 x 5

In [5]:
mdata.var_names

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'guide0', 'guide1',
       'guide2', 'guide3', 'guide4'],
      dtype='object')

In [6]:
perturbvi.PERTURBVI.setup_mudata(
    mdata,
    size_factor_key = 'lib_size',
    modalities={
        "rna_layer": rna_key,
        "perturbation_layer": perturb_key,
    },
)

model = perturbvi.PERTURBVI(mdata)
model.summary_stats

n_batch: 1
n_cells: 1000
n_extra_categorical_covs: 0
n_extra_continuous_covs: 0
n_perturbations: 5
n_vars: 10

In [7]:
model = perturbvi.PERTURBVI(mdata)
# pyro.render_model(model.module.model, (torch.tensor(rna_counts),))

In [12]:
model.train(max_epochs=100, train_size=1, lr=0.1)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/trainer.py:1609: PossibleUserWarning: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower 

Epoch 100/100: 100%|██████████| 100/100 [00:01<00:00, 52.26it/s, v_num=1, elbo_train=4.91e+3]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████| 100/100 [00:01<00:00, 51.98it/s, v_num=1, elbo_train=4.91e+3]


In [14]:
for k,v in pyro.get_param_store().items():
    print(k, v)

log_var_mean.mu tensor([-7.3858, -7.4123, -7.4689, -7.4615, -7.4888, -7.4093, -7.4331, -7.4260,
        -7.4003, -7.4822], requires_grad=True)
log_var_disp.mu tensor([0.2101, 0.2546, 0.2383, 0.2505, 0.2733, 0.2333, 0.2706, 0.2143, 0.2196,
        0.2687], requires_grad=True)


With myst it is possible to link in the text cell of a notebook such as this one the documentation of a function or a class.

Let's take as an example the function {func}`perturbvi.pp.basic_preproc`. 
You can see that by clicking on the text, the link redirects to the API documentation of the function. 
Check the raw markdown of this cell to understand how this is specified.

This works also for any package listed by `intersphinx`. Go to `docs/conf.py` and look for the `intersphinx_mapping` variable. 
There, you will see a list of packages (that this package is dependent on) for which this functionality is supported. 

For instance, we can link to the class {class}`anndata.AnnData`, to the attribute {attr}`anndata.AnnData.obs` or the method {meth}`anndata.AnnData.write`.

Again, check the raw markdown of this cell to see how each of these links are specified.

You can read more about this in the [intersphinx page](https://www.sphinx-doc.org/en/master/usage/extensions/intersphinx.html) and the [myst page](https://myst-parser.readthedocs.io/en/v0.15.1/syntax/syntax.html#roles-an-in-line-extension-point).